# StylePops Bootstrap Pipeline — İP-1 → İP-3

## ⚠️ ÖNEMLİ: `.ipynb` dosyasını bilgisayardan yüklemeyin!

Colab **sadece notebook dosyasını** alır; `data/`, `scripts/` vb. gelmez.
Bu yüzden sol panelde sadece `.ipynb` görürsünüz — bu normal.

**Doğru yol:** Aşağıdaki **"0. GitHub Clone"** hücresini çalıştırın.
Tüm proje `/content/StylePops_Modules/` altına iner.

**Colab'ı şu linkten açın (önerilen):**
https://colab.research.google.com/github/valueisinvalid/StylePops_Modules/blob/main/notebooks/StylePops_Bootstrap_Pipeline.ipynb

---

**Bu notebook:**
1. Bootstrap envanteri yükler (200 parça)
2. İP-1: Özellik vektörü üretir (SBERT + renk + kumaş)
3. İP-2: Termal model (`hedef_Clo`, `efektif_Clo`) + LightGBM v0.1
4. İP-3: MCDM kombin sıralaması (`Rank = Final_Skor − |hedef_Clo − Total_Clo|`)

## 0. GitHub'dan Projeyi Clone Et (İLK ÇALIŞTIRILACAK HÜCRE)

Bu hücreden sonra sol panelde **Files → StylePops_Modules → data** klasörünü görmelisiniz.

In [ ]:
import os

REPO_URL = "https://github.com/valueisinvalid/StylePops_Modules.git"
PROJECT_ROOT = "/content/StylePops_Modules"

if os.path.isdir(PROJECT_ROOT):
    !cd {PROJECT_ROOT} && git pull
else:
    !git clone {REPO_URL} {PROJECT_ROOT}

assert os.path.isdir(PROJECT_ROOT), f"Klasör bulunamadı: {PROJECT_ROOT}"
print("Proje kökü:", PROJECT_ROOT)
!ls {PROJECT_ROOT}

In [ ]:
# Dosyaları doğrula
!ls -R {PROJECT_ROOT}/data

## 1. Kurulum (pip)

In [ ]:
!pip install -q sentence-transformers lightgbm scikit-learn pandas numpy

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path(PROJECT_ROOT) / "data"
LOOKUPS = DATA / "lookups"
BOOTSTRAP = DATA / "bootstrap"

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

fabric_props = load_json(LOOKUPS / "fabric_properties.json")
target_clo = load_json(LOOKUPS / "target_clo_points.json")
coverage_ratios = load_json(LOOKUPS / "coverage_ratios.json")
season_templates = load_json(LOOKUPS / "season_templates.json")

garments_payload = load_json(BOOTSTRAP / "garments_200.json")
garments = {g["id"]: g for g in garments_payload["garments"]}
combos_df = pd.read_csv(BOOTSTRAP / "combinations_200.csv")

MATERIALS = fabric_props["materials"]
THERMAL_CATS = fabric_props["thermal_categories"]
CLO_POINTS = target_clo["target_clo_points"]
WEATHER_SCENARIOS = target_clo["weather_scenarios"]

print(f"Parça sayısı: {len(garments)}")
print(f"Kombin sayısı: {len(combos_df)}")
combos_df.head(3)

## 2. İP-1 — Veri İşleme ve Özellik Vektörü

In [ ]:
def parse_fabric_composition(fabrics):
    """Kumaş kompozisyonu ayrıştırma (İP-1)."""
    total = sum(f["pct"] for f in fabrics)
    return [(f["material"], f["pct"] / total) for f in fabrics]


def weighted_property(fabrics, prop="clo", layers=None):
    """ic_Clo veya ret_degeri — katmanlı giysilerde ağırlıklı ortalama."""
    if layers:
        num = sum(
            layer["weight"] * sum(
                (f["pct"] / 100) * MATERIALS[f["material"]][prop]
                for f in layer["fabric_composition"]
            )
            for layer in layers
        )
        den = sum(layer["weight"] for layer in layers)
        return num / den if den else 0.0

    parsed = parse_fabric_composition(fabrics)
    return sum(w * MATERIALS[m][prop] for m, w in parsed)


def effective_clo(garment, use_category_fallback=False):
    if use_category_fallback:
        ic = THERMAL_CATS[garment["thermal_category"]]["clo"]
    else:
        ic = weighted_property(garment["fabric_composition"], "clo", garment.get("layers"))
    return ic * garment["coverage_ratio"]


def effective_ret(garment, use_category_fallback=False):
    if use_category_fallback:
        return THERMAL_CATS[garment["thermal_category"]]["ret"]
    return weighted_property(garment["fabric_composition"], "ret", garment.get("layers"))


# Örnek
g = garments["G001"]
print(g["name"], "-> efektif_Clo:", round(effective_clo(g), 4))

In [ ]:
from sentence_transformers import SentenceTransformer

# GPU varsa otomatik kullanır
sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

descriptions = [g["description"] for g in garments.values()]
ids = list(garments.keys())
style_embeddings = sbert.encode(descriptions, show_progress_bar=True, batch_size=32)
style_vectors = {gid: emb for gid, emb in zip(ids, style_embeddings)}

print("SBERT boyutu:", style_embeddings.shape)

In [ ]:
def garment_feature_vector(garment):
    """771 boyutlu özellik vektörünün bootstrap yaklaşımı."""
    style = style_vectors[garment["id"]]
    lab = garment["color_lab"]
    color = np.array([lab["L"], lab["a"], lab["b"]])
    thermal = np.array([
        effective_clo(garment),
        effective_ret(garment),
        garment["coverage_ratio"],
    ])
    # Kumaş one-hot (14 malzeme)
    mat_names = list(MATERIALS.keys())
    mat_vec = np.zeros(len(mat_names))
    for f in garment["fabric_composition"]:
        if f["material"] in mat_names:
            mat_vec[mat_names.index(f["material"])] = f["pct"] / 100
    return np.concatenate([style, color, thermal, mat_vec])


def combo_feature_vector(piece_ids):
    vecs = [garment_feature_vector(garments[pid]) for pid in piece_ids if pid]
    return np.mean(vecs, axis=0)

sample = combo_feature_vector(["G001", "G079", "G040"])
print("Kombin özellik boyutu:", sample.shape)

## 3. İP-2 — Termal Modelleme

In [ ]:
def wind_chill_c(T_c, V_kmh):
    if T_c > 10 or V_kmh < 4.8:
        return T_c
    v_ms = V_kmh / 3.6
    return 13.12 + 0.6215 * T_c - 11.37 * (v_ms ** 0.16) + 0.3965 * T_c * (v_ms ** 0.16)


def heat_index_c(T_c, RH):
    if T_c < 27:
        return T_c
    Tf = T_c * 9 / 5 + 32
    HI = (
        -42.379 + 2.04901523 * Tf + 10.14333127 * RH
        - 0.22475541 * Tf * RH - 0.00683783 * Tf ** 2
        - 0.05481717 * RH ** 2 + 0.00122874 * Tf ** 2 * RH
        + 0.00085282 * Tf * RH ** 2 - 0.00000199 * Tf ** 2 * RH ** 2
    )
    return (HI - 32) * 5 / 9


def apparent_temperature(T_hava, RH, V_ruzgar):
    if T_hava <= 10 and V_ruzgar >= 5:
        return round(wind_chill_c(T_hava, V_ruzgar), 2)
    if T_hava >= 27:
        return round(heat_index_c(T_hava, RH), 2)
    return round(T_hava, 2)


def interpolate_hedef_clo(T):
    pts = sorted(CLO_POINTS, key=lambda p: p["T_celsius"])
    if T <= pts[0]["T_celsius"]:
        return pts[0]["clo"]
    if T >= pts[-1]["T_celsius"]:
        return pts[-1]["clo"]
    for i in range(len(pts) - 1):
        t1, c1 = pts[i]["T_celsius"], pts[i]["clo"]
        t2, c2 = pts[i + 1]["T_celsius"], pts[i + 1]["clo"]
        if t1 <= T <= t2:
            return round(c1 + (T - t1) * (c2 - c1) / (t2 - t1), 4)
    return pts[-1]["clo"]


def weather_to_hedef_clo(T_hava, RH, V_ruzgar):
    T_hissedilen = apparent_temperature(T_hava, RH, V_ruzgar)
    return T_hissedilen, interpolate_hedef_clo(T_hissedilen)

# Test: kış senaryosu
T_h, hedef = weather_to_hedef_clo(2, 65, 25)
print(f"T_hissedilen={T_h}°C -> hedef_Clo={hedef}")

## 4. İP-2 — Uyarlayıcı Puanlama (LightGBM v0.1)

Etiketler boşsa kural tabanlı fallback devreye girer (Risk 1/4 B Planı).

In [ ]:
def rule_based_aesthetic_score(piece_ids):
    """Risk 4 B Planı — renk uyumu + temel parça bonusları."""
    pieces = [garments[pid] for pid in piece_ids if pid]
    if not pieces:
        return 2.5
    labs = np.array([[p["color_lab"]["L"], p["color_lab"]["a"], p["color_lab"]["b"]] for p in pieces])
    color_std = np.std(labs, axis=0).mean()
    color_score = max(1.0, 5.0 - color_std / 8)
    categories = {p["category"] for p in pieces}
    bonus = 0.3 if "top" in categories and "bottom" in categories else 0
    return min(5.0, color_score + bonus)


def build_training_data(combos_df):
    X, y_est, y_th = [], [], []
    for _, row in combos_df.iterrows():
        if pd.isna(row.get("aesthetic_score")) or row.get("aesthetic_score") == "":
            continue
        piece_ids = row["piece_ids"].split("|")
        X.append(combo_feature_vector(piece_ids))
        y_est.append(float(row["aesthetic_score"]))
        if not pd.isna(row.get("thermal_score")) and row.get("thermal_score") != "":
            y_th.append(float(row["thermal_score"]))
    return np.array(X), np.array(y_est), np.array(y_th) if y_th else None


X_train, y_est_train, y_th_train = build_training_data(combos_df)
lgbm_aesthetic = None
USE_LGBM = len(y_est_train) >= 20  # en az 20 etiketli kombin

if USE_LGBM:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score

    X_tr, X_te, y_tr, y_te = train_test_split(X_train, y_est_train, test_size=0.2, random_state=42)
    lgbm_aesthetic = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
    lgbm_aesthetic.fit(X_tr, y_tr)
    r2 = r2_score(y_te, lgbm_aesthetic.predict(X_te))
    print(f"LightGBM estetik R² = {r2:.3f}")
    if r2 < 0.70:
        print("R² < 0.70 — kural tabanlı fallback kullanılacak.")
        lgbm_aesthetic = None
else:
    print(f"Etiketli kombin: {len(y_est_train)} — LightGBM atlandı, kural tabanlı skorlama aktif.")
    print("combinations_200.csv dosyasındaki aesthetic_score sütununu doldurun.")

In [ ]:
def predict_aesthetic(piece_ids):
    if lgbm_aesthetic is not None:
        return float(lgbm_aesthetic.predict(combo_feature_vector(piece_ids).reshape(1, -1))[0])
    return rule_based_aesthetic_score(piece_ids)

## 5. İP-3 — Çok Kriterli Optimizasyon (MCDM)

In [ ]:
def thermal_bonuses_penalties(piece_ids, hedef_clo):
    """Kural tabanlı ceza/bonus (İP-3 Adım 2)."""
    bonus, penalty = 0.0, 0.0
    pieces = [garments[pid] for pid in piece_ids if pid]
    total_clo = sum(effective_clo(p) for p in pieces)
    avg_ret = np.mean([effective_ret(p) for p in pieces])
    avg_coverage = np.mean([p["coverage_ratio"] for p in pieces])

    if hedef_clo > 1.0:  # soğuk
        if avg_coverage < 0.6:
            penalty += 1.5
        if total_clo > 0.25:
            bonus += 0.5
        if any(p["subcategory"] in {"tank_top", "shorts"} for p in pieces):
            penalty += 2.0
    elif hedef_clo < 0.5:  # sıcak
        if avg_ret > 7:
            penalty += 1.0
        if any(p["subcategory"] in {"shorts", "tank_top", "dress_short"} for p in pieces):
            bonus += 0.5
        if total_clo > 0.6:
            penalty += 1.5

    return bonus, penalty, total_clo


def score_combination(piece_ids, hedef_clo):
    skor_estetik = predict_aesthetic(piece_ids)
    bonus, penalty, total_clo = thermal_bonuses_penalties(piece_ids, hedef_clo)
    final_skor = skor_estetik + bonus - penalty
    rank = final_skor - abs(hedef_clo - total_clo)
    return {
        "piece_ids": piece_ids,
        "skor_estetik": round(skor_estetik, 3),
        "bonus": round(bonus, 3),
        "penalty": round(penalty, 3),
        "final_skor": round(final_skor, 3),
        "total_Clo_C": round(total_clo, 4),
        "delta_Clo": round(abs(hedef_clo - total_clo), 4),
        "rank": round(rank, 3),
    }

In [ ]:
def generate_candidate_combos(inventory_ids, n_candidates=500, seed=42):
    """Envanterden rastgele aday kombinasyonlar üret."""
    rng = np.random.default_rng(seed)
    inv = [garments[i] for i in inventory_ids]
    tops = [g["id"] for g in inv if g["category"] in {"top", "dress"}]
    bottoms = [g["id"] for g in inv if g["category"] == "bottom"]
    outers = [g["id"] for g in inv if g["category"] == "outer"]
    candidates = []
    for _ in range(n_candidates):
        top = rng.choice(tops)
        gtop = garments[top]
        if gtop["category"] == "dress":
            outer = rng.choice(outers) if outers and rng.random() < 0.2 else None
            candidates.append([top] + ([outer] if outer else []))
        else:
            bottom = rng.choice(bottoms) if bottoms else None
            outer = rng.choice(outers) if outers and rng.random() < 0.4 else None
            ids = [top] + ([bottom] if bottom else []) + ([outer] if outer else [])
            candidates.append(ids)
    return candidates


def recommend_outfit(scenario_key, inventory_ids=None, top_k=5):
    scenario = WEATHER_SCENARIOS[scenario_key]
    T_hissedilen, hedef_clo = weather_to_hedef_clo(
        scenario["T_hava"], scenario["RH_nem"], scenario["V_ruzgar"]
    )
    if inventory_ids is None:
        inventory_ids = list(garments.keys())

    candidates = generate_candidate_combos(inventory_ids)
    scored = [score_combination(c, hedef_clo) for c in candidates]
    ranked = sorted(scored, key=lambda x: x["rank"], reverse=True)[:top_k]

    print(f"Senaryo: {scenario.get('label_tr', scenario_key)}")
    print(f"T_hissedilen={T_hissedilen}°C | hedef_Clo={hedef_clo}")
    print("-" * 60)
    for i, r in enumerate(ranked, 1):
        names = " + ".join(garments[pid]["name"] for pid in r["piece_ids"])
        print(f"#{i} Rank={r['rank']} | {names}")
        print(f"   estetik={r['skor_estetik']} | Clo={r['total_Clo_C']} | ΔClo={r['delta_Clo']}")
    return ranked

## 6. Demo — 4 Mevsim Senaryosu

In [ ]:
for scenario in ["kis_soguk_ruzgarli", "sonbahar_serin", "ilkbahar_ilik", "yaz_sicak"]:
    recommend_outfit(scenario, top_k=3)
    print()

## 7. Bootstrap Kombinleri Değerlendir (Etiketleme Sonrası)

In [ ]:
def evaluate_labeled_combos(combos_df):
    results = []
    for _, row in combos_df.iterrows():
        piece_ids = row["piece_ids"].split("|")
        hedef = row["hedef_Clo"]
        scored = score_combination(piece_ids, hedef)
        scored["combo_id"] = row["combo_id"]
        scored["season"] = row["season"]
        if not pd.isna(row.get("aesthetic_score")) and row.get("aesthetic_score") != "":
            scored["label_aesthetic"] = float(row["aesthetic_score"])
        results.append(scored)
    return pd.DataFrame(results)

eval_df = evaluate_labeled_combos(combos_df)
eval_df.groupby("season")["rank", "delta_Clo"].mean().round(3)

## 8. Sonuçları Kaydet

In [ ]:
OUT = Path(PROJECT_ROOT) / "outputs"
OUT.mkdir(exist_ok=True)

eval_df.to_csv(OUT / "bootstrap_eval_results.csv", index=False)
print(f"Kaydedildi: {OUT / 'bootstrap_eval_results.csv'}")

if lgbm_aesthetic is not None:
    import joblib
    joblib.dump(lgbm_aesthetic, OUT / "lgbm_aesthetic_v0.1.joblib")
    print(f"Model kaydedildi: {OUT / 'lgbm_aesthetic_v0.1.joblib'}")